# 0. Environment Setup and Installations

In [ ]:
# 0. Environment Setup and Installations

# 1. Mount Google Drive to access your files
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)

# 2. Extract the balanced dataset directly to the working directory
print("Unzipping Dataset... this might take a few minutes.")
!rm -rf /content/dataset
os.makedirs("/content/dataset", exist_ok=True)
zip_path = "/content/drive/MyDrive/Deepfake_Project/DFDC_Img.zip"
!unzip -q "{zip_path}" -d "/content/dataset"
print("Unzip complete!")

# 3. Install necessary libraries
print("Installing required libraries...")
!pip install -q transformers accelerate bitsandbytes torch torchvision Pillow tqdm pandas scikit-learn matplotlib seaborn
print("Installations complete! Ready for Data Extraction.")

Mounted at /content/drive
Unzipping Dataset... this might take a few minutes.
Unzip complete!
Installing required libraries...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 41.2 MB/s eta 0:00:00
Installations complete! Ready for Data Extraction.


# 1 Copy Img paths from Drive + dfdc split generator

In [ ]:
# 1. DFDC Split Generator (Run once - saved to Drive to prevent data leakage)
import os, glob, json, random

SPLIT_JSON_PATH = '/content/drive/MyDrive/Deepfake_Project/dfdc_split.json'
SEED = 42
TRAIN_PER_CLASS = 6232

# If the split already exists on Drive, just load it
if os.path.exists(SPLIT_JSON_PATH):
    with open(SPLIT_JSON_PATH, 'r') as f:
        split = json.load(f)
    print("Loaded existing split from Drive.")

else:
    # Collect all paths from both train and validation folders into one pool
    base = '/content/dataset'
    fake_paths = (
        glob.glob(os.path.join(base, 'train', 'fake', '**', '*.*'), recursive=True) +
        glob.glob(os.path.join(base, 'validation', 'fake', '**', '*.*'), recursive=True)
    )
    real_paths = (
        glob.glob(os.path.join(base, 'train', 'real', '**', '*.*'), recursive=True) +
        glob.glob(os.path.join(base, 'validation', 'real', '**', '*.*'), recursive=True)
    )
    fake_paths = [p for p in fake_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]
    real_paths = [p for p in real_paths if p.lower().endswith(('.png', '.jpg', '.jpeg'))]

    print("Data Loading Summary:")
    print(f"Found {len(real_paths)} REAL images.")
    print(f"Found {len(fake_paths)} FAKE images.")
    print(f"Total: {len(real_paths) + len(fake_paths)} images accurately loaded and mapped.")

    # Shuffle with fixed seed to ensure reproducibility
    random.seed(SEED)
    random.shuffle(real_paths)
    random.shuffle(fake_paths)

    # Split: 10% balanced for training, rest for evaluation
    split = {
        'train': {
            'real': real_paths[:TRAIN_PER_CLASS],
            'fake': fake_paths[:TRAIN_PER_CLASS]
        },
        'test': {
            'real': real_paths[TRAIN_PER_CLASS:],
            'fake': fake_paths[TRAIN_PER_CLASS:]
        }
    }

    # Save to Drive - this file is the single source of truth for all future runs
    with open(SPLIT_JSON_PATH, 'w') as f:
        json.dump(split, f)
    print(f"Split saved to Drive: {SPLIT_JSON_PATH}")

# Summary
print(f"\nSplit Summary:")
print(f"Train — REAL: {len(split['train']['real'])} | FAKE: {len(split['train']['fake'])}")
print(f"Test  — REAL: {len(split['test']['real'])} | FAKE: {len(split['test']['fake'])}")
print(f"Total Test images: {len(split['test']['real']) + len(split['test']['fake'])}")

Loaded existing split from Drive.

Split Summary:
Train — REAL: 6232 | FAKE: 6232
Test  — REAL: 20496 | FAKE: 91687
Total Test images: 112183


# 2. Model Initialization

In [ ]:
from transformers import InstructBlipProcessor, InstructBlipForConditionalGeneration
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# Load Processor
processor = InstructBlipProcessor.from_pretrained("Salesforce/instructblip-vicuna-7b")

# Load Model in bfloat16 for stability and memory efficiency
model = InstructBlipForConditionalGeneration.from_pretrained(
    "Salesforce/instructblip-vicuna-7b",
    torch_dtype=torch.bfloat16
).to(device)

# --- FREEZING WEIGHTS (PEFT Strategy) ---
# We freeze the Vision Encoder and the LLM, and only train the Q-Former
for name, param in model.named_parameters():
    if "qformer" in name or "query_tokens" in name or "language_projection" in name:
        param.requires_grad = True
    else:
        param.requires_grad = False

# Verify freezing: print trainable parameters count
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable_params:,} ({100 * trainable_params / all_params:.2f}%)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


processor_config.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.24k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.51k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/104k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

Trainable parameters: 188,834,560 (2.39%)


# 3. Dataset Loading

In [ ]:
# 3. Dataset Loading

import os
import json
import torch
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import multiprocessing

class TrainDeepfakeDataset(Dataset):
    def __init__(self, data_frame, processor):
        self.data = data_frame
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label_str = row['label']
        prompt = "is this photo real?"

        inputs = self.processor(images=image, text=prompt, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        prompt_len = inputs["input_ids"].shape[0]

        answer_tokens = self.processor.tokenizer(" " + label_str, return_tensors="pt", add_special_tokens=False).input_ids.squeeze(0)
        answer_len = answer_tokens.shape[0]

        inputs["input_ids"] = torch.cat([inputs["input_ids"], answer_tokens])
        inputs["attention_mask"] = torch.cat([inputs["attention_mask"], torch.ones(answer_len, dtype=torch.long)])

        prompt_labels = torch.full((prompt_len,), -100, dtype=torch.long)
        inputs["labels"] = torch.cat([prompt_labels, answer_tokens])
        return inputs

# Load split from Drive
SPLIT_JSON_PATH = '/content/drive/MyDrive/Deepfake_Project/dfdc_split.json'
with open(SPLIT_JSON_PATH, 'r') as f:
    split = json.load(f)

# Build train DataFrame from JSON
train_paths  = split['train']['real'] + split['train']['fake']
train_labels = ['Yes'] * len(split['train']['real']) + ['No'] * len(split['train']['fake'])

train_df = pd.DataFrame({'image_path': train_paths, 'label': train_labels})
print(f"Training images (DFDC): {len(train_df)}")

train_dataset = TrainDeepfakeDataset(train_df, processor)
optimal_workers = min(8, multiprocessing.cpu_count())

train_dataloader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    num_workers=optimal_workers,
    pin_memory=True,
    prefetch_factor=2
)
print("Train DataLoader ready!")

Training images (DFDC): 12464
Train DataLoader ready!


# 4. Fine tuning Training Loop

In [ ]:
import torch
import os
from torch.optim import AdamW
from tqdm import tqdm
import json

# --- 1. Optimizer Setup ---
optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-5,
    weight_decay=0.05,
    betas=(0.9, 0.999)
)

# --- 2. Training Loop ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.train()

print("Starting Fine-Tuning Quality Pass...")
training_loss_history = []
loop = tqdm(train_dataloader, leave=True, desc="Training")

for batch in loop:
    optimizer.zero_grad()

    batch = {k: v.to(device) for k, v in batch.items() if isinstance(v, torch.Tensor)}

    outputs = model(**batch)
    loss = outputs.loss

    loss.backward()

    torch.nn.utils.clip_grad_norm_(filter(lambda p: p.requires_grad, model.parameters()), max_norm=1.0)

    optimizer.step()

    current_loss = loss.item()
    training_loss_history.append(current_loss)
    loop.set_postfix(loss=f"{current_loss:.4f}")

# --- 3. Save the Model ---
save_dir = "/content/drive/MyDrive/Deepfake_Project/DFDC_Weights_slow_5e"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "qformer_finetuned_DFDC_slow_5e.pth")

torch.save(model.qformer.state_dict(), save_path)
print(f"\nSuccess: Fine-tuned Q-Former weights saved to {save_path}")

# --- 4. save loss history ---

with open(os.path.join(save_dir, "loss_history.json"), "w") as f:
    json.dump(training_loss_history, f)

Starting Fine-Tuning Quality Pass...


Training: 100%|██████████| 3116/3116 [10:14<00:00,  5.07it/s, loss=0.0097]



Success: Fine-tuned Q-Former weights saved to /content/drive/MyDrive/Deepfake_Project/DFDC_Weights_slow_5e/qformer_finetuned_DFDC_slow_5e.pth
